# Figure S28 — Tephra thickness versus distance


## 1. Imports and shared 00_Setup

This standalone notebook uses Arial, 8-point ordinary text, full axes boxes, light gridlines, 300 dpi PNG export, and editable PDF/SVG text. The code is linear and contains no custom functions.


In [1]:
from pathlib import Path
import hashlib

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from PIL import Image

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = 5.2
BASE_TEXT_SIZE_PT = 8.0
AXIS_LABEL_SIZE_PT = 8.0
LEGEND_SIZE_PT = 7.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":AXIS_LABEL_SIZE_PT,
    "xtick.labelsize":BASE_TEXT_SIZE_PT, "ytick.labelsize":BASE_TEXT_SIZE_PT,
    "legend.fontsize":LEGEND_SIZE_PT, "figure.facecolor":"white",
    "axes.facecolor":"white", "axes.spines.top":True, "axes.spines.right":True,
    "axes.linewidth":0.5, "lines.linewidth":0.75,
    "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})


## 2. Paths and source workbook

The figure reads only the public S3 workbook sheet Hoblitt_1987. S1 and S2 are checked as part of the shared supplemental package but are not used for this figure.


In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
DATA_DIRECTORY = WORK_ROOT / "DataSheets" / "Updated_Oct20"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_S28_Hoblitt"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_S28_Hoblitt_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = DATA_DIRECTORY / "Dechert_S1_AllData.xlsx"
S2_PATH = DATA_DIRECTORY / "Dechert_S2_Standards.xlsx"
S3_PATH = DATA_DIRECTORY / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH, S2_PATH, S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_WORKBOOK = S3_PATH
SOURCE_SHEET = "Hoblitt_1987"
SOURCE_HASH_BEFORE = hashlib.sha256(SOURCE_WORKBOOK.read_bytes()).hexdigest()


## 3. Read the established Hoblitt series


In [3]:
x_column = "Distance_km"
series = {
    "GlacierPeakG_cm": {"label":"Glacier Peak G", "color":"#ff1493", "marker":"s", "markersize":7, "linewidth":1.8},
    "Mazama_cm": {"label":"Mazama/Crater Lake", "color":"#8B4513", "marker":"^", "markersize":7, "linewidth":1.8},
    "StHelens_1980_cm": {"label":"Mount St. Helens 1980", "color":"#708238", "marker":"o", "markersize":7, "linewidth":1.8},
}
requested_columns = [x_column, *series]
source_df = pd.read_excel(SOURCE_WORKBOOK, sheet_name=SOURCE_SHEET, usecols=requested_columns)
source_df = source_df.loc[:, requested_columns].copy()
plot_data = {}
retained_counts = {}
for column, style in series.items():
    numeric_data = source_df[[x_column, column]].apply(pd.to_numeric, errors="coerce")
    retained = numeric_data.dropna(subset=[x_column, column]).copy()
    if not retained[x_column].is_monotonic_increasing:
        retained = retained.sort_values(x_column, kind="stable")
    if (retained[column] <= 0).any():
        raise ValueError(f"{column} contains non-positive values and cannot use a log y-axis.")
    plot_data[column] = retained
    retained_counts[column] = len(retained)


## 4. Plot and export Figure S28


In [4]:
figure, axis = plt.subplots(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))
for column, style in series.items():
    data = plot_data[column]
    axis.plot(
        data[x_column], data[column], label=style["label"], color=style["color"],
        marker=style["marker"], markersize=style["markersize"], linewidth=style["linewidth"],
        markeredgecolor=style["color"], markeredgewidth=0.5,
        solid_capstyle="round", solid_joinstyle="round", zorder=3,
    )
axis.set_xlabel("Distance (km)")
axis.set_ylabel("Tephra Thickness (cm)")
axis.set_xlim(0, 1100)
axis.set_yscale("log")
axis.grid(True, which="major", color="0.82", linewidth=0.45, alpha=0.55, zorder=0)
axis.set_axisbelow(True)
axis.legend(loc="upper right", frameon=True, edgecolor="black", framealpha=1.0)
for spine in axis.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(0.5)
figure.subplots_adjust(left=0.105, right=0.975, bottom=0.115, top=0.975)

FIGURE_STEM = "Figure_S28_Tephra_Thickness_Distance"
EXPORT_PATHS = {}
for output_format in ["pdf", "png", "svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{output_format}"
    save_options = {"format":output_format, "facecolor":"white", "transparent":False}
    if output_format == "png":
        save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path, **save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise IOError(f"Expected export was not created: {output_path}")
    EXPORT_PATHS[output_format] = output_path
plt.close(figure)


## 5. Validate the source, plot, and exports


In [5]:
assert SOURCE_SHEET in pd.ExcelFile(SOURCE_WORKBOOK).sheet_names
assert retained_counts == {"GlacierPeakG_cm":6, "Mazama_cm":16, "StHelens_1980_cm":15}
assert axis.get_yscale() == "log"
assert tuple(round(value, 8) for value in axis.get_xlim()) == (0.0, 1100.0)
assert tuple(round(value, 8) for value in figure.get_size_inches()) == (FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN)
assert all(spine.get_visible() for spine in axis.spines.values())
assert all(path.is_file() and path.stat().st_size > 0 for path in EXPORT_PATHS.values())
with Image.open(EXPORT_PATHS["png"]) as image:
    assert image.size == (2175, 1560)
svg_text = EXPORT_PATHS["svg"].read_text(encoding="utf-8", errors="replace")
assert "<text" in svg_text and "<image" not in svg_text
assert b"/Type3" not in EXPORT_PATHS["pdf"].read_bytes()
assert hashlib.sha256(SOURCE_WORKBOOK.read_bytes()).hexdigest() == SOURCE_HASH_BEFORE
print("Figure S28 complete.")
print("Retained points:", retained_counts)
print("Outputs:", {key:str(value) for key,value in EXPORT_PATHS.items()})


Figure S28 complete.
Retained points: {'GlacierPeakG_cm': 6, 'Mazama_cm': 16, 'StHelens_1980_cm': 15}
Outputs: {'pdf': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S28_Hoblitt/Figure_S28_Hoblitt_Final/Figure_S28_Tephra_Thickness_Distance.pdf', 'png': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S28_Hoblitt/Figure_S28_Hoblitt_Final/Figure_S28_Tephra_Thickness_Distance.png', 'svg': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_S28_Hoblitt/Figure_S28_Hoblitt_Final/Figure_S28_Tephra_Thickness_Distance.svg'}
